# ANOVA 方差分析完整实现示例

## 目标

掌握单因素、双因素和重复测量 ANOVA 的应用。

## 内容

- 单因素ANOVA：检验一个因素多个水平的均值差异
- 双因素ANOVA：分析两个因素的主效应和交互效应
- 重复测量ANOVA：分析同一被试在不同条件下的差异
- 事后检验：Tukey HSD、Bonferroni等
- 效应量：eta-squared、partial eta-squared
- 假设检验：正态性、方差齐性、球形假设

## 1. 数据准备

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.anova import AnovaRM
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子
np.random.seed(42)

print("库导入完成")

## 2. 单因素ANOVA（One-way ANOVA）

### 2.1 数据示例：三种教学方法的效果

In [ ]:
# 生成三种教学方法的数据
np.random.seed(42)

# 方法A：传统讲授
method_A = np.random.normal(75, 8, 30)

# 方法B：互动讨论
method_B = np.random.normal(80, 8, 30)

# 方法C：项目学习
method_C = np.random.normal(82, 8, 30)

# 创建DataFrame
data = pd.DataFrame({
    'score': np.concatenate([method_A, method_B, method_C]),
    'method': ['方法A'] * 30 + ['方法B'] * 30 + ['方法C'] * 30
})

print("数据描述：")
print(data.groupby('method')['score'].describe())
print("\n各组均值：")
print(data.groupby('method')['score'].mean())

### 2.2 执行单因素ANOVA

In [ ]:
# 使用scipy执行单因素ANOVA
f_stat, p_value = stats.f_oneway(method_A, method_B, method_C)

print("单因素ANOVA结果（scipy）")
print("=" * 40)
print(f"F统计量: {f_stat:.4f}")
print(f"p值: {p_value:.4f}")

# 决策
alpha = 0.05
if p_value < alpha:
    print(f"\n结论：由于 p = {p_value:.4f} < {alpha}，拒绝原假设")
    print("至少有一种方法与其他方法的效果有显著差异")
else:
    print(f"\n结论：由于 p = {p_value:.4f} >= {alpha}，不拒绝原假设")
    print("没有证据表明三种方法的效果有差异")

### 2.3 使用statsmodels获取ANOVA表

In [ ]:
# 使用statsmodels
model = ols('score ~ C(method)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("ANOVA表：")
print(anova_table)

# 计算效应量 eta-squared
ss_between = anova_table.loc['C(method)', 'sum_sq']
ss_total = anova_table['sum_sq'].sum()
eta_squared = ss_between / ss_total

print(f"\n效应量 eta-squared = {eta_squared:.4f}")
print("效应量解释：")
if eta_squared < 0.01:
    print("  小效应")
elif eta_squared < 0.06:
    print("  中等效应")
elif eta_squared < 0.14:
    print("  大效应")
else:
    print("  极大效应")

### 2.4 单因素ANOVA可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 箱线图
sns.boxplot(x='method', y='score', data=data, ax=axes[0])
axes[0].set_title('各组分数分布（箱线图）')
axes[0].set_xlabel('教学方法')
axes[0].set_ylabel('分数')
axes[0].grid(True, alpha=0.3, axis='y')

# 小提琴图
sns.violinplot(x='method', y='score', data=data, ax=axes[1])
axes[1].set_title('各组分数分布密度（小提琴图）')
axes[1].set_xlabel('教学方法')
axes[1].set_ylabel('分数')
axes[1].grid(True, alpha=0.3, axis='y')

# 均值条形图
sns.barplot(x='method', y='score', data=data, ax=axes[2], ci=95, capsize=0.1)
axes[2].set_title('各组均值与95%置信区间')
axes[2].set_xlabel('教学方法')
axes[2].set_ylabel('分数')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("各组描述统计：")
print(data.groupby('method')['score'].agg(['mean', 'std', 'count']).round(2))

## 3. 事后检验（Post-hoc Tests）

当ANOVA结果显著时，需要进行事后检验确定具体哪些组之间存在差异。

In [ ]:
# Tukey HSD事后检验
tukey = pairwise_tukeyhsd(data['score'], data['method'])

print("Tukey HSD事后检验结果：")
print("=" * 60)
print(tukey)

print("\n解释：")
print("- reject=True 表示两组之间存在显著差异")
print("- meandiff 是两组均值差")
print("- p-adj 是调整后的p值")
print("- lower/upper 是95%置信区间的上下限")

### 3.1 事后检验可视化

In [ ]:
# 获取Tukey检验结果
from statsmodels.stats.multicomp import MultiComparison

mc = MultiComparison(data['score'], data['method'])
tukey_result = mc.tukeyhsd()

# 提取数据用于可视化
results = []
for i in range(len(tukey_result._results_table.data)):
    row = tukey_result._results_table.data[i]
    if i == 0:
        continue  # 跳过表头
    results.append({
        'group1': row[0],
        'group2': row[1],
        'meandiff': float(row[2]),
        'lower': float(row[3]),
        'upper': float(row[4]),
        'p-adj': float(row[5]),
        'reject': row[6]
    })

df_results = pd.DataFrame(results)

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))

for idx, row in df_results.iterrows():
    color = 'red' if row['reject'] else 'gray'
    alpha = 0.8 if row['reject'] else 0.3
    
    ax.plot([0, 1], [idx, idx], color='black', linewidth=1)
    ax.plot([0.5], [idx], 'o', color=color, markersize=10, alpha=alpha)
    
    ax.text(0.5, idx + 0.1, f"{row['meandiff']:.2f}", 
            ha='center', va='bottom', fontsize=10)
    ax.text(1.1, idx, f"{row['group1']} - {row['group2']}", 
            ha='left', va='center', fontsize=10)
    
    # 添加置信区间
    ax.errorbar([0.5], [idx], 
                xerr=[[row['meandiff'] - row['lower']], [row['upper'] - row['meandiff']]],
                fmt='none', color='blue', linewidth=2, alpha=0.6)

ax.set_xlim(-0.1, 2.5)
ax.set_ylim(-0.5, len(df_results) - 0.5)
ax.set_yticks([])
ax.set_xlabel('均值差')
ax.set_title('Tukey HSD事后检验结果')
ax.grid(True, alpha=0.3, axis='x')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)

# 添加图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', alpha=0.8, label='显著差异 (p<0.05)'),
    Patch(facecolor='gray', alpha=0.3, label='不显著 (p>=0.05)')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

## 4. 假设检验

### 4.1 正态性检验

In [ ]:
print("1. 正态性检验 (Shapiro-Wilk)")
print("=" * 60)

for method, group_data in data.groupby('method')['score']:
    stat, p = stats.shapiro(group_data)
    print(f"{method}:")
    print(f"  统计量: {stat:.4f}")
    print(f"  p值: {p:.4f}")
    print(f"  结论: {'拒绝正态性假设' if p < 0.05 else '无法拒绝正态性假设'}")
    print()

# Q-Q图
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (method, group_data) in enumerate(data.groupby('method')['score']):
    stats.probplot(group_data, dist="norm", plot=axes[idx])
    axes[idx].set_title(f'{method} Q-Q图')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 方差齐性检验

In [ ]:
print("2. 方差齐性检验")
print("=" * 60)

# Levene检验
stat_levene, p_levene = stats.levene(method_A, method_B, method_C)
print("Levene检验:")
print(f"  统计量: {stat_levene:.4f}")
print(f"  p值: {p_levene:.4f}")
print(f"  结论: {'方差不齐' if p_levene < 0.05 else '方差齐性假设成立'}")

# Bartlett检验
stat_bartlett, p_bartlett = stats.bartlett(method_A, method_B, method_C)
print(f"\nBartlett检验:")
print(f"  统计量: {stat_bartlett:.4f}")
print(f"  p值: {p_bartlett:.4f}")
print(f"  结论: {'方差不齐' if p_bartlett < 0.05 else '方差齐性假设成立'}")

# 可视化方差
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 方差条形图
variances = data.groupby('method')['score'].var()
axes[0].bar(variances.index, variances.values, color=['lightblue', 'lightgreen', 'lightcoral'])
axes[0].set_ylabel('方差')
axes[0].set_title('各组方差')
axes[0].grid(True, alpha=0.3, axis='y')

# 标准差条形图
stds = data.groupby('method')['score'].std()
axes[1].bar(stds.index, stds.values, color=['lightblue', 'lightgreen', 'lightcoral'])
axes[1].set_ylabel('标准差')
axes[1].set_title('各组标准差')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. 双因素ANOVA（Two-way ANOVA）

### 5.1 数据示例：教学方法 × 学生性别

In [ ]:
# 生成双因素数据
np.random.seed(42)

n_per_cell = 15

# 四个单元格：方法A-男, 方法A-女, 方法B-男, 方法B-女
data_2way = pd.DataFrame({
    'score': np.concatenate([
        np.random.normal(78, 5, n_per_cell),  # 方法A-男
        np.random.normal(82, 5, n_per_cell),  # 方法A-女
        np.random.normal(80, 5, n_per_cell),  # 方法B-男
        np.random.normal(84, 5, n_per_cell)   # 方法B-女
    ]),
    'method': ['方法A'] * n_per_cell + ['方法A'] * n_per_cell + 
              ['方法B'] * n_per_cell + ['方法B'] * n_per_cell,
    'gender': ['男'] * n_per_cell + ['女'] * n_per_cell + 
              ['男'] * n_per_cell + ['女'] * n_per_cell
})

print("双因素数据描述：")
print(data_2way.groupby(['method', 'gender'])['score'].describe())

print("\n各组合均值：")
print(data_2way.pivot_table(values='score', index='method', columns='gender', aggfunc='mean'))

### 5.2 执行双因素ANOVA

In [ ]:
# 执行双因素ANOVA（包含交互效应）
model_2way = ols('score ~ C(method) * C(gender)', data=data_2way).fit()
anova_table_2way = sm.stats.anova_lm(model_2way, typ=2)

print("双因素ANOVA表（含交互效应）：")
print(anova_table_2way)

# 计算偏eta-squared
def partial_eta_squared(anova_table, effect):
    """计算偏eta-squared"""
    ss_effect = anova_table.loc[effect, 'sum_sq']
    ss_error = anova_table.loc['Residual', 'sum_sq']
    return ss_effect / (ss_effect + ss_error)

print("\n偏eta-squared (Partial eta-squared):")
for effect in ['C(method)', 'C(gender)', 'C(method):C(gender)']:
    p_eta = partial_eta_squared(anova_table_2way, effect)
    print(f"  {effect}: {p_eta:.4f}")

### 5.3 双因素ANOVA可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 交互作用图
sns.pointplot(x='method', y='score', hue='gender', data=data_2way, 
              markers=['o', 's'], linestyles=['-', '--'], ax=axes[0],
              ci=95, capsize=0.1)
axes[0].set_title('交互作用图：教学方法 × 性别')
axes[0].set_xlabel('教学方法')
axes[0].set_ylabel('分数')
axes[0].legend(title='性别')
axes[0].grid(True, alpha=0.3, axis='y')

# 箱线图（分面）
sns.boxplot(x='method', y='score', hue='gender', data=data_2way, ax=axes[1])
axes[1].set_title('各组合分数分布')
axes[1].set_xlabel('教学方法')
axes[1].set_ylabel('分数')
axes[1].legend(title='性别')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. 重复测量ANOVA（Repeated Measures ANOVA）

### 6.1 数据示例：4个时间点的测量

In [ ]:
# 生成重复测量数据
np.random.seed(42)

n_subjects = 20
n_timepoints = 4

data_rm = pd.DataFrame({
    'subject': np.repeat(range(n_subjects), n_timepoints),
    'time': np.tile(['T1', 'T2', 'T3', 'T4'], n_subjects),
    'score': np.concatenate([
        np.random.normal(70, 5, n_subjects),  # T1
        np.random.normal(75, 5, n_subjects),  # T2
        np.random.normal(80, 5, n_subjects),  # T3
        np.random.normal(85, 5, n_subjects)   # T4
    ])
})

print("重复测量数据示例（前5个被试）：")
print(data_rm.head(20))

print("\n各时间点描述统计：")
print(data_rm.groupby('time')['score'].describe())

### 6.2 执行重复测量ANOVA

In [ ]:
# 执行重复测量ANOVA
aovrm = AnovaRM(data_rm, 'score', 'subject', within=['time'])
res = aovrm.fit()

print("重复测量ANOVA结果：")
print(res)

# 如果交互效应显著，可以进行事后检验
# 这里我们手动计算Tukey检验
from scipy.stats import tukey_hsd

scores_by_time = [data_rm[data_rm['time'] == t]['score'].values 
                 for t in ['T1', 'T2', 'T3', 'T4']]
tukey_result = tukey_hsd(*scores_by_time)

print("\n各时间点两两比较 (Tukey HSD):")
print(tukey_result)

### 6.3 重复测量ANOVA可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 折线图（显示个体轨迹）
for subject in range(n_subjects):
    subject_data = data_rm[data_rm['subject'] == subject]
    axes[0].plot(subject_data['time'], subject_data['score'], 
                 'o-', alpha=0.3, linewidth=1)

# 添加均值线
means = data_rm.groupby('time')['score'].mean()
axes[0].plot(means.index, means.values, 'ko-', linewidth=3, markersize=10, 
             label='均值')

axes[0].set_xlabel('时间点')
axes[0].set_ylabel('分数')
axes[0].set_title('个体轨迹与均值')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# 箱线图
sns.boxplot(x='time', y='score', data=data_rm, ax=axes[1])
axes[1].set_xlabel('时间点')
axes[1].set_ylabel('分数')
axes[1].set_title('各时间点分数分布')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. 方差不齐时的替代方法

### 7.1 Welch ANOVA

In [ ]:
# 生成方差不齐的数据
np.random.seed(42)

group1 = np.random.normal(70, 3, 30)   # 小方差
group2 = np.random.normal(75, 8, 30)   # 大方差
group3 = np.random.normal(78, 5, 30)   # 中方差

# 标准ANOVA
f_stat, p_value = stats.f_oneway(group1, group2, group3)

# Welch ANOVA
def welch_anova(*groups):
    """执行Welch ANOVA"""
    k = len(groups)
    n_i = np.array([len(g) for g in groups])
    mean_i = np.array([np.mean(g) for g in groups])
    var_i = np.array([np.var(g, ddof=1) for g in groups])
    
    # 权重
    w_i = n_i / var_i
    
    # 加权均值
    y_bar = np.sum(w_i * mean_i) / np.sum(w_i)
    
    # F统计量
    numerator = np.sum(w_i * (mean_i - y_bar)**2) / (k - 1)
    denominator = 1 + 2 * (k - 2) / (k**2 - 1) * np.sum((1 - w_i / np.mean(w_i))**2 / (n_i - 1))
    
    f_welch = numerator / denominator
    
    # 自由度
    df1 = k - 1
    df2 = k**2 - 1 / np.sum((1 - w_i / np.mean(w_i))**2 / (n_i - 1))
    
    p_value = 1 - stats.f.cdf(f_welch, df1, df2)
    
    return f_welch, p_value, df1, df2

f_welch, p_welch, df1, df2 = welch_anova(group1, group2, group3)

print("标准ANOVA vs Welch ANOVA对比")
print("=" * 50)
print(f"各组方差: {np.var(group1, ddof=1):.2f}, {np.var(group2, ddof=1):.2f}, {np.var(group3, ddof=1):.2f}")
print(f"\n标准ANOVA: F = {f_stat:.4f}, p = {p_value:.4f}")
print(f"Welch ANOVA: F({df1:.1f}, {df2:.1f}) = {f_welch:.4f}, p = {p_welch:.4f}")

# 方差齐性检验
stat_levene, p_levene = stats.levene(group1, group2, group3)
print(f"\nLevene检验: p = {p_levene:.4f}")
print(f"结论: {'方差不齐，推荐使用Welch ANOVA' if p_levene < 0.05 else '方差齐性假设成立，可用标准ANOVA'}")

## 8. 残差分析

In [ ]:
# 使用单因素ANOVA的模型进行残差分析
residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 残差直方图
axes[0, 0].hist(residuals, bins=15, edgecolor='black', alpha=0.7, color='lightblue')
axes[0, 0].set_xlabel('残差')
axes[0, 0].set_ylabel('频数')
axes[0, 0].set_title('残差分布直方图')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Q-Q图
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('残差Q-Q图')
axes[0, 1].grid(True, alpha=0.3)

# 残差 vs 拟合值
axes[1, 0].scatter(fitted, residuals, alpha=0.6)
axes[1, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('拟合值')
axes[1, 0].set_ylabel('残差')
axes[1, 0].set_title('残差 vs 拟合值')
axes[1, 0].grid(True, alpha=0.3)

# 标准化残差
std_residuals = (residuals - np.mean(residuals)) / np.std(residuals)
axes[1, 1].scatter(range(len(std_residuals)), std_residuals, alpha=0.6)
axes[1, 1].axhline(y=2, color='r', linestyle='--', linewidth=1)
axes[1, 1].axhline(y=-2, color='r', linestyle='--', linewidth=1)
axes[1, 1].set_xlabel('观测值')
axes[1, 1].set_ylabel('标准化残差')
axes[1, 1].set_title('标准化残差')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 检查残差的正态性
stat, p = stats.shapiro(residuals)
print(f"残差正态性检验 (Shapiro-Wilk): p = {p:.4f}")
print(f"结论: {'残差不正态' if p < 0.05 else '残差符合正态分布'}")

## 9. 总结

### ANOVA类型

| 类型 | 用途 | 特点 |
|------|------|------|
| 单因素ANOVA | 比较一个因素多个水平 | 前提：正态性、独立性、方差齐性 |
| 双因素ANOVA | 分析两个因素的主效应和交互效应 | 可以分析交互作用 |
| 重复测量ANOVA | 分析同一被试在不同条件下的差异 | 控制个体差异，提高功效 |

### 关键要点

1. **前提条件检查：**
   - 正态性：Shapiro-Wilk检验、Q-Q图
   - 方差齐性：Levene检验、Bartlett检验
   - 独立性：通过研究设计保证
   - 球形假设（重复测量）：Mauchly检验

2. **事后检验：**
   - Tukey HSD：控制家族错误率
   - Bonferroni校正：简单但保守
   - Dunnett检验：与对照组比较

3. **效应量：**
   - eta-squared：因素解释的变异比例
   - partial eta-squared：消除其他因素后的效应量

4. **报告完整信息：**
   - ANOVA表（F值、p值）
   - 效应量
   - 事后检验结果
   - 假设检验结果

5. **替代方法：**
   - 方差不齐：Welch ANOVA
   - 非正态：Kruskal-Wallis检验（非参数）
   - 不满足球形假设：多层线性模型、广义估计方程